# Анализ календарных аномалий

Данные загружены в `01_load_data.ipynb`. Здесь: чистка, расчёт
дневных доходностей, проверка различий по дням недели.

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

In [4]:
prices = pd.read_csv(RAW / "moex_prices.csv", parse_dates=["TRADEDATE"])

print(prices.shape)
prices.head()

(69457, 24)


,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2014-01-06,Сбербанк,SBER,22830,3.154470e+09,100.20,98.62,100.31,98.63,...,99.54,99.54,3.156739e+09,3.156739e+09,3.156739e+09,NaN,3,SUR,-2.23,NaN
1,TQBR,2014-01-08,Сбербанк,SBER,35633,4.179939e+09,99.10,97.85,99.41,98.20,...,98.65,98.65,4.182984e+09,4.182984e+09,4.182984e+09,NaN,3,SUR,-0.73,NaN
2,TQBR,2014-01-09,Сбербанк,SBER,41567,4.518389e+09,98.44,97.69,98.77,97.97,...,98.25,98.25,4.526414e+09,4.526414e+09,4.526414e+09,NaN,3,SUR,-0.19,NaN
3,TQBR,2014-01-10,Сбербанк,SBER,38198,5.109679e+09,97.87,97.52,99.41,99.41,...,98.45,98.45,5.113831e+09,5.113831e+09,5.113831e+09,NaN,3,SUR,1.22,NaN
4,TQBR,2014-01-13,Сбербанк,SBER,29942,6.191507e+09,99.30,99.04,100.35,100.24,...,99.78,99.78,6.191738e+09,6.191738e+09,6.191738e+09,NaN,3,SUR,1.06,NaN


In [5]:
prices.dtypes

BOARDID                               str
TRADEDATE                  datetime64[us]
SHORTNAME                             str
SECID                                 str
NUMTRADES                           int64
VALUE                             float64
OPEN                              float64
LOW                               float64
HIGH                              float64
LEGALCLOSEPRICE                   float64
WAPRICE                           float64
CLOSE                             float64
VOLUME                              int64
MARKETPRICE2                      float64
MARKETPRICE3                      float64
ADMITTEDQUOTE                     float64
MP2VALTRD                         float64
MARKETPRICE3TRADESVALUE           float64
ADMITTEDVALUE                     float64
WAVAL                             float64
TRADINGSESSION                      int64
CURRENCYID                            str
TRENDCLSPR                        float64
TRADE_SESSION_DATE                

In [6]:
before = len(prices)

prices = prices[prices["CLOSE"].notna()]
prices = prices[["TRADEDATE", "SECID", "CLOSE", "VOLUME"]]
prices = prices.sort_values(["SECID", "TRADEDATE"]).reset_index(drop=True)

print(f"было {before}, стало {len(prices)}, удалено {before - len(prices)}")
prices.head()

было 69457, стало 69005, удалено 452


,TRADEDATE,SECID,CLOSE,VOLUME
0,2014-01-06,AFLT,83.24,2490000
1,2014-01-08,AFLT,83.41,2345100
2,2014-01-09,AFLT,82.90,2751400
3,2014-01-10,AFLT,82.83,3849100
4,2014-01-13,AFLT,87.87,7404300


In [7]:
print(prices["CLOSE"].isna().sum())

0


In [8]:
prices["RETURN"] = prices.groupby("SECID")["CLOSE"].pct_change()

prices[["TRADEDATE", "SECID", "CLOSE", "RETURN"]].head()

,TRADEDATE,SECID,CLOSE,RETURN
0,2014-01-06,AFLT,83.24,NaN
1,2014-01-08,AFLT,83.41,0.002042
2,2014-01-09,AFLT,82.90,-0.006114
3,2014-01-10,AFLT,82.83,-0.000844
4,2014-01-13,AFLT,87.87,0.060848


In [9]:
print(prices["RETURN"].isna().sum())

25


In [10]:
prices["DOW"] = prices["TRADEDATE"].dt.dayofweek
prices["MONTH"] = prices["TRADEDATE"].dt.month
prices["YEAR"] = prices["TRADEDATE"].dt.year

prices[["TRADEDATE", "DOW", "MONTH", "YEAR"]].head()

,TRADEDATE,DOW,MONTH,YEAR
0,2014-01-06,0,1,2014
1,2014-01-08,2,1,2014
2,2014-01-09,3,1,2014
3,2014-01-10,4,1,2014
4,2014-01-13,0,1,2014


In [11]:
prices["DOW"].value_counts().sort_index()

DOW
0    13530
1    13805
2    13819
3    13869
4    13781
5      201
Name: count, dtype: int64

In [12]:
saturdays = prices[prices["DOW"] == 5]

print(saturdays["TRADEDATE"].dt.year.value_counts().sort_index())
print()
print(saturdays["TRADEDATE"].sort_values().unique()[:20])

TRADEDATE
2016    21
2018    63
2021    22
2024    70
2025    25
Name: count, dtype: int64

<DatetimeArray>
['2016-02-20 00:00:00', '2018-04-28 00:00:00', '2018-06-09 00:00:00',
 '2018-12-29 00:00:00', '2021-02-20 00:00:00', '2024-04-27 00:00:00',
 '2024-11-02 00:00:00', '2024-12-28 00:00:00', '2025-11-01 00:00:00']
Length: 9, dtype: datetime64[us]


In [13]:
dates = saturdays["TRADEDATE"].sort_values().unique()
for d in dates:
    print(pd.Timestamp(d).strftime("%Y-%m-%d"))

2016-02-20
2018-04-28
2018-06-09
2018-12-29
2021-02-20
2024-04-27
2024-11-02
2024-12-28
2025-11-01


### Рабочие субботы

В данных нашлось 9 суббот с торгами (201 наблюдение): переносы
выходных перед праздниками — 20.02.2016, 28.04.2018, 09.06.2018,
29.12.2018, 20.02.2021, 27.04.2024, 02.11.2024, 28.12.2024, 01.11.2025.

Решение: исключаем. Причины: (1) 201 наблюдение против ~14 000
у каждого буднего дня — статистически бессмысленная группа;
(2) торги в такие дни идут с пониженной активностью, то есть
это не обычный торговый день, и различие объяснялось бы не
днём недели, а режимом торгов.

In [14]:
before = len(prices)
prices = prices[prices["DOW"] < 5].reset_index(drop=True)
print(f"удалено {before - len(prices)} строк по рабочим субботам")

удалено 201 строк по рабочим субботам


In [15]:
prices["RETURN"].describe()

count    68779.000000
mean         0.069618
std         17.784189
min         -0.989839
25%         -0.009513
50%         -0.000138
75%          0.009820
max       4662.823382
Name: RETURN, dtype: float64

In [19]:
extremes = prices.reindex(prices["RETURN"].abs().sort_values(ascending=False).index)
print(extremes.shape)
extremes[["TRADEDATE", "SECID", "CLOSE", "RETURN"]].head(15)

(68804, 8)


,TRADEDATE,SECID,CLOSE,RETURN
67869,2024-07-15,VTBR,92.9500,4662.823382
15720,2015-01-20,IRAO,0.7641,106.317416
14882,2024-04-08,GMKN,152.9600,-0.989839
43228,2025-03-27,PLZL,1867.6000,-0.901793
61998,2026-04-17,T,326.2600,-0.897916
67288,2022-02-24,VTBR,0.0201,-0.412538
43628,2022-03-29,POSI,1095.8000,0.398418
43627,2022-03-28,POSI,783.6000,0.398287
57488,2022-02-24,SMLT,2336.0000,-0.397938
17506,2022-02-24,IRAO,2.0430,-0.375897


### Сплиты убираем, рыночные движения оставляем

##### Порог поставим на ±50%. Движения такого размера за день на ликвидных голубых фишках не случаются даже в кризис — максимум реального падения в таблице 41%.

In [20]:
SPLIT_THRESHOLD = 0.5

splits = prices["RETURN"].abs() > SPLIT_THRESHOLD
print(f"помечено как сплиты: {splits.sum()}")
print(prices.loc[splits, ["TRADEDATE", "SECID", "RETURN"]])

prices.loc[splits, "RETURN"] = None
print(f"осталось доходностей: {prices['RETURN'].notna().sum()}")

помечено как сплиты: 5
       TRADEDATE SECID       RETURN
14882 2024-04-08  GMKN    -0.989839
15720 2015-01-20  IRAO   106.317416
43228 2025-03-27  PLZL    -0.901793
61998 2026-04-17     T    -0.897916
67869 2024-07-15  VTBR  4662.823382
осталось доходностей: 68774


In [22]:
prices["RETURN"].describe()

count    68774.000000
mean         0.000319
std          0.021170
min         -0.412538
25%         -0.009512
50%         -0.000138
75%          0.009819
max          0.398418
Name: RETURN, dtype: float64

### Технические выбросы: сплиты

Пять наблюдений с |доходностью| > 50% — не рыночные движения,
а изменения номинала:
- VTBR 15.07.2024 (+466200%) — обратный сплит 5000:1
- IRAO 20.01.2015 (+10632%) — консолидация
- GMKN 08.04.2024, PLZL 27.03.2025, T 17.04.2026 (~-90%) — дробление

Решение: доходность в эти дни помечена как неопределённая (NaN).
Строки не удалялись, чтобы не создать фиктивную доходность
между соседними днями.

Порог 50% выбран по данным: максимальное реальное движение
в выборке — обвал 24.02.2022 (-41%), все технические
аномалии превышают 90%.

Эффект: среднее упало с 0.0696 до 0.00032, std с 17.8 до 0.021.
Квартили не изменились — выбросы искажали только моменты
распределения, но не его форму.